# 手撕反向传播（Backpropagation）

> 本文件原为空，按"详细解析 + 骨架"补全。请按注释提示实现各节点反向。

## 1. 原理
反向传播 = 在**计算图**上沿**反拓扑序**用**链式法则**逐节点计算 $\partial L/\partial$ 参数。
- 前向：按依赖顺序算出每个中间值并缓存。
- 反向：从 $L$ 出发，对每个节点 $v$，把上游所有 $\partial L/\partial u$ 沿 $u\to v$ 的局部雅可比回传累加，得到 $\partial L/\partial v$。

## 2. 示例网络
两层 MLP：$h=\sigma(W_1 x+b_1)\to y=\sigma(W_2 h+b_2)\to L=\tfrac12\|y-\hat y\|^2$，$\sigma$ 为 sigmoid。
记号：$z_1=W_1 x+b_1,\ h=\sigma(z_1),\ z_2=W_2 h+b_2,\ y=\sigma(z_2)$。

## 3. 反向推导（手算）
- $\delta_y = y-\hat y$（MSE 对 $y$ 的梯度）
- $\delta_{z_2}=\delta_y\odot\sigma'(z_2)=\delta_y\odot y(1-y)$
- $\partial L/\partial W_2=\delta_{z_2} h^T,\quad \partial L/\partial b_2=\delta_{z_2}$
- $\delta_h=W_2^T\delta_{z_2}$
- $\delta_{z_1}=\delta_h\odot\sigma'(z_1)=\delta_h\odot h(1-h)$
- $\partial L/\partial W_1=\delta_{z_1} x^T,\quad \partial L/\partial b_1=\delta_{z_1}$

## 4. 考察点
- 拓扑序与缓存（前向存什么、反向用什）
- sigmoid 导数 $\sigma(1-\sigma)$ 用前向值而非重算
- 与 autograd 对拍验证

In [ ]:
import torch
import torch.nn.functional as F

# 网络配置
din, dh, dout = 4, 3, 2
torch.manual_seed(0)
W1 = torch.randn(dh, din, requires_grad=True)
b1 = torch.randn(dh, requires_grad=True)
W2 = torch.randn(dout, dh, requires_grad=True)
b2 = torch.randn(dout, requires_grad=True)
x = torch.randn(din)
y_hat = torch.randn(dout)

def sigmoid(z):
    return 1.0 / (1.0 + torch.exp(-z))

# ---- 前向（缓存中间值）----
# TODO: z1 = W1 @ x + b1 ; h = sigmoid(z1) ; z2 = W2 @ h + b2 ; y = sigmoid(z2) ; L = 0.5*||y-y_hat||^2
raise NotImplementedError

# ---- 反向（按推导手写各梯度）----
# TODO:
# delta_y = y - y_hat
# delta_z2 = delta_y * y * (1 - y)
# grad_W2 = outer(delta_z2, h) ; grad_b2 = delta_z2
# delta_h = W2.T @ delta_z2
# delta_z1 = delta_h * h * (1 - h)
# grad_W1 = outer(delta_z1, x) ; grad_b1 = delta_z1
raise NotImplementedError

# ---- 与 autograd 对拍 ----
# TODO: 用 autograd 重算一遍，断言 (grad_W1 - W1.grad).abs().max() < 1e-6 等
raise NotImplementedError

## 小结
- 反向的本质是"沿计算图反拓扑序传播梯度"，每个节点只需自己的局部导数 + 上游梯度。
- 手写时**前向要缓存**反向要用的值（如 $z_1,z_2,h,y$），这正是 activation checkpointing 节省的对象。
- 面试常要求对一个小网络写出所有参数的解析梯度，务必练熟 sigmoid/ReLU/MSE/CE 的局部导数。

## ✅ 测试验证

In [ ]:
# 验证手动反向传播与 autograd 一致
import torch

# 简单测试: y = x^2, dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
assert abs(x.grad.item() - 6.0) < 1e-6, f"grad should be 6.0, got {x.grad}"

# 复合测试: y = sigmoid(x), dy/dx = y(1-y)
x = torch.tensor(0.5, requires_grad=True)
y = torch.sigmoid(x)
y.backward()
expected = y.item() * (1 - y.item())
assert abs(x.grad.item() - expected) < 1e-6, f"sigmoid grad mismatch: {x.grad} vs {expected}"

# 链式法则: y = (x^2 + 1)^3, dy/dx = 3(x^2+1)^2 * 2x
x = torch.tensor(2.0, requires_grad=True)
y = (x**2 + 1)**3
y.backward()
expected = 3 * (4+1)**2 * 2 * 2  # 3 * 25 * 4 = 300
assert abs(x.grad.item() - expected) < 1e-4, f"chain rule mismatch: {x.grad} vs {expected}"

print("✅ Backpropagation 测试通过: 与 autograd 一致，链式法则正确")
